**Intrusion Detection System**

Problem Statement : Intrusion Detection System (IDS): Create a machine learning-based
IDS that can identify and respond to threats in real time.


ML

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.svm import SVC
from sklearn.metrics import classification_report
import joblib


In [21]:
df = pd.read_csv("final_train.csv")
num_rows_with_nulls = df.isnull().any(axis=1).sum()
print(f"Number of rows with at least one null value: {num_rows_with_nulls}")


Number of rows with at least one null value: 0


In [22]:
X = df.drop(columns=["level", "binary_attack"])
y = df["binary_attack"]

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

categorical_cols = ["protocol_type", "service", "flag"]
numerical_cols = X.columns.difference(categorical_cols).tolist()

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

svm_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", SVC(kernel="rbf", C=100.0))
])
svm_pipeline.fit(x_train, y_train)
y_pred = svm_pipeline.predict(x_test)
print(classification_report(y_test, y_pred))
joblib.dump(svm_pipeline, "svm_rbf_kdd_model.pkl")
print("✅ SVM RBF model saved as 'svm_rbf_kdd_model.pkl'")

              precision    recall  f1-score   support

    abnormal       1.00      1.00      1.00     11773
      normal       1.00      1.00      1.00     13422

    accuracy                           1.00     25195
   macro avg       1.00      1.00      1.00     25195
weighted avg       1.00      1.00      1.00     25195

✅ SVM RBF model saved as 'svm_rbf_kdd_model.pkl'


In [23]:
from sklearn.metrics import accuracy_score
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")

Accuracy: 0.9960706489382815


In [24]:
import pandas as pd
df1 = pd.read_csv('final_test.csv')
df_sample = df1.head(10)
y_true = df_sample['binary_attack'].values
df_sample_features.to_csv('logfile12.csv', index=False)
print("logfile12.csv saved and y_true extracted.")


logfile12.csv saved and y_true extracted.


In [25]:
log_df = pd.read_csv("logfile12.csv")

X_log = log_df.drop(columns=["level", "binary_attack"], errors="ignore")

svm_pipeline = joblib.load("svm_rbf_kdd_model.pkl")

log_predictions = svm_pipeline.predict(X_log)

output_df = log_df.copy()
output_df["true_attack"] = y_true
output_df["predicted_attack"] = log_predictions
output_df.to_csv("log_with_predictions.csv", index=False)

print("Predictions added to 'log_with_predictions.csv'")


Predictions added to 'log_with_predictions.csv'


In [26]:
accuracy = accuracy_score(y_true, log_predictions)
print(f"✅ Accuracy: {accuracy:.4f}")

✅ Accuracy: 0.7000


In [27]:
df_1 = pd.read_csv("log_with_predictions.csv")
df_1


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,level,true_attack,predicted_attack
0,0,tcp,private,REJ,0,0,0,0,0,0,...,0.06,0.00,0.00,0.00,0.00,1.00,1.00,21,abnormal,abnormal
1,0,tcp,private,REJ,0,0,0,0,0,0,...,0.06,0.00,0.00,0.00,0.00,1.00,1.00,21,abnormal,abnormal
2,2,tcp,ftp_data,SF,12983,0,0,0,0,0,...,0.04,0.61,0.02,0.00,0.00,0.00,0.00,21,normal,normal
3,0,icmp,eco_i,SF,20,0,0,0,0,0,...,0.00,1.00,0.28,0.00,0.00,0.00,0.00,15,abnormal,abnormal
4,1,tcp,telnet,RSTO,0,15,0,0,0,0,...,0.17,0.03,0.02,0.00,0.00,0.83,0.71,11,abnormal,normal
5,0,tcp,http,SF,267,14515,0,0,0,0,...,0.00,0.01,0.03,0.01,0.00,0.00,0.00,21,normal,normal
6,0,tcp,smtp,SF,1022,387,0,0,0,0,...,0.72,0.00,0.00,0.00,0.00,0.72,0.04,21,normal,normal
7,0,tcp,telnet,SF,129,174,0,0,0,0,...,0.00,0.00,0.00,0.01,0.01,0.02,0.02,15,abnormal,normal
8,0,tcp,http,SF,327,467,0,0,0,0,...,0.00,0.01,0.03,0.00,0.00,0.00,0.00,21,normal,normal
9,0,tcp,ftp,SF,26,157,0,0,0,0,...,0.08,0.02,0.00,0.00,0.00,0.00,0.00,7,abnormal,normal


**DL**

In [28]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import numpy as np

df = pd.read_csv("final_train.csv")
X = df.drop(columns=["level", "binary_attack"])
y = df["binary_attack"].map({'normal': 0, 'abnormal': 1})
categorical_cols = ["protocol_type", "service", "flag"]
numerical_cols = X.columns.difference(categorical_cols).tolist()
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

X_processed = preprocessor.fit_transform(X)

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X_processed, y.values, test_size=0.2, random_state=42
)
X_train_tensor = torch.tensor(X_train_np.toarray() if hasattr(X_train_np, "toarray") else X_train_np, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_np.toarray() if hasattr(X_test_np, "toarray") else X_test_np, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_np, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test_np, dtype=torch.float32).unsqueeze(1)
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

class IntrusionNet(nn.Module):
    def __init__(self, input_dim):
        super(IntrusionNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

model = IntrusionNet(input_dim=X_train_tensor.shape[1])
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        output = model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

torch.save(model.state_dict(), "intrusion_net.pth")
joblib.dump(preprocessor, "dl_preprocessor.pkl")
print("✅ DL model and preprocessor saved.")


Epoch 1/20, Loss: 0.0584
Epoch 2/20, Loss: 0.0301
Epoch 3/20, Loss: 0.0260
Epoch 4/20, Loss: 0.0254
Epoch 5/20, Loss: 0.0229
Epoch 6/20, Loss: 0.0224
Epoch 7/20, Loss: 0.0205
Epoch 8/20, Loss: 0.0198
Epoch 9/20, Loss: 0.0193
Epoch 10/20, Loss: 0.0189
Epoch 11/20, Loss: 0.0204
Epoch 12/20, Loss: 0.0191
Epoch 13/20, Loss: 0.0183
Epoch 14/20, Loss: 0.0183
Epoch 15/20, Loss: 0.0178
Epoch 16/20, Loss: 0.0175
Epoch 17/20, Loss: 0.0171
Epoch 18/20, Loss: 0.0165
Epoch 19/20, Loss: 0.0158
Epoch 20/20, Loss: 0.0171
✅ DL model and preprocessor saved.


In [30]:
import joblib

log_df = pd.read_csv("logfile12.csv")
X_log = log_df.drop(columns=["level", "binary_attack"], errors="ignore")

preprocessor = joblib.load("dl_preprocessor.pkl")
X_log_processed = preprocessor.transform(X_log)
X_log_tensor = torch.tensor(
    X_log_processed.toarray() if hasattr(X_log_processed, "toarray") else X_log_processed,
    dtype=torch.float32
)

model = IntrusionNet(input_dim=X_log_tensor.shape[1])
model.load_state_dict(torch.load("intrusion_net.pth"))
model.eval()

with torch.no_grad():
    predictions = model(X_log_tensor).squeeze()
    predicted_labels = (predictions >= 0.5).int().numpy()


mapped_labels = ["abnormal" if p == 1 else "normal" for p in predicted_labels]

output_df = log_df.copy()
output_df["binary_attack"] = mapped_labels
output_df.to_csv("log_with_predictions_dl.csv", index=False)

print("✅ Deep Learning predictions saved to 'log_with_predictions_dl.csv'")
output_df

✅ Deep Learning predictions saved to 'log_with_predictions_dl.csv'


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,level,binary_attack
0,0,tcp,private,REJ,0,0,0,0,0,0,...,0.04,0.06,0.00,0.00,0.00,0.00,1.00,1.00,21,abnormal
1,0,tcp,private,REJ,0,0,0,0,0,0,...,0.00,0.06,0.00,0.00,0.00,0.00,1.00,1.00,21,abnormal
2,2,tcp,ftp_data,SF,12983,0,0,0,0,0,...,0.61,0.04,0.61,0.02,0.00,0.00,0.00,0.00,21,normal
3,0,icmp,eco_i,SF,20,0,0,0,0,0,...,1.00,0.00,1.00,0.28,0.00,0.00,0.00,0.00,15,abnormal
4,1,tcp,telnet,RSTO,0,15,0,0,0,0,...,0.31,0.17,0.03,0.02,0.00,0.00,0.83,0.71,11,normal
5,0,tcp,http,SF,267,14515,0,0,0,0,...,1.00,0.00,0.01,0.03,0.01,0.00,0.00,0.00,21,normal
6,0,tcp,smtp,SF,1022,387,0,0,0,0,...,0.11,0.72,0.00,0.00,0.00,0.00,0.72,0.04,21,normal
7,0,tcp,telnet,SF,129,174,0,0,0,0,...,1.00,0.00,0.00,0.00,0.01,0.01,0.02,0.02,15,normal
8,0,tcp,http,SF,327,467,0,0,0,0,...,1.00,0.00,0.01,0.03,0.00,0.00,0.00,0.00,21,normal
9,0,tcp,ftp,SF,26,157,0,0,0,0,...,0.50,0.08,0.02,0.00,0.00,0.00,0.00,0.00,7,normal


**QML**

In [1]:
!pip install pennylane --quiet


In [2]:
import pennylane as qml
from pennylane import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
from sklearn.decomposition import PCA
df = pd.read_csv("final_train.csv")

X = df.drop(columns=["level", "binary_attack"])
y = df["binary_attack"].map({'normal': 0, 'abnormal': 1})

categorical_cols = ["protocol_type", "service", "flag"]
numerical_cols = X.columns.difference(categorical_cols).tolist()

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

X_processed = preprocessor.fit_transform(X)
X_processed = X_processed.toarray() if hasattr(X_processed, "toarray") else X_processed

X_processed = X_processed[:10000]
y = y[:10000].values


pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_processed)

X_train_tensor = torch.tensor(X_pca, dtype=torch.float32)
y_train_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=2, shuffle=True)


n_qubits = 2
dev = qml.device("default.qubit", wires=n_qubits)


@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    qml.templates.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.templates.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))

class QMLNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.classical = nn.Linear(2, 2)
        self.q_params = nn.Parameter(0.01 * torch.randn((2, n_qubits)))

    def forward(self, x):
        x = self.classical(x)
        outputs = []
        for i in range(x.shape[0]):
            out = quantum_circuit(x[i], self.q_params)
            outputs.append(out)
        return torch.stack(outputs).unsqueeze(1)

model = QMLNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.BCELoss()

epochs = 15
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = torch.sigmoid(model(xb)).float()
        yb = yb.float()
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss / len(train_loader):.4f}")


FileNotFoundError: [Errno 2] No such file or directory: 'final_train.csv'

In [ ]:
import joblib

joblib.dump(preprocessor, "qml_preprocessor.pkl")
joblib.dump(pca, "qml_pca.pkl")

print("Preprocessor and PCA saved as .pkl files")

In [33]:
def save_model(model, path="qml_model.pth"):
    torch.save(model.state_dict(), path)
    print(f"Model saved to {path}")
save_model(model, "qml_model.pth")


Model saved to qml_model.pth


In [34]:

def save_quantum_params(q_params, path="quantum_params.pth"):
    torch.save(q_params, path)
    print(f"Quantum parameters saved to {path}")
save_quantum_params(model.q_params, "quantum_params.pth")
def load_quantum_params(path="quantum_params.pth"):
    return torch.load(path)

model.q_params = load_quantum_params("quantum_params.pth")


Quantum parameters saved to quantum_params.pth


In [35]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

df_log = pd.read_csv("logfile12.csv")
df_original = pd.read_csv("final_test.csv").head(10)
y_true = df_original["binary_attack"].map({'normal': 0, 'abnormal': 1}).values

X_log = df_log.drop(columns=["level", "binary_attack"], errors="ignore")
X_log_processed = preprocessor.transform(X_log)
X_log_processed = X_log_processed.toarray() if hasattr(X_log_processed, "toarray") else X_log_processed

X_log_pca = pca.transform(X_log_processed)
X_log_tensor = torch.tensor(X_log_pca, dtype=torch.float32)


model.load_state_dict(torch.load("qml_model.pth"))
model.eval()

with torch.no_grad():
    outputs = torch.sigmoid(model(X_log_tensor)).squeeze()
    y_pred = (outputs >= 0.5).int().numpy()


accuracy = accuracy_score(y_true, y_pred)
print(f"Quantum Model Accuracy: {accuracy:.4f}")
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=["normal", "abnormal"]))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))


result_df = df_log.copy()
result_df["true_attack"] = ["abnormal" if y == 1 else "normal" for y in y_true]
result_df["predicted_attack"] = ["abnormal" if y == 1 else "normal" for y in y_pred]
result_df.to_csv("log_with_qml_predictions.csv", index=False)
print(" Results saved to 'log_with_qml_predictions.csv'")


Quantum Model Accuracy: 0.6000

Classification Report:
               precision    recall  f1-score   support

      normal       0.50      1.00      0.67         4
    abnormal       1.00      0.33      0.50         6

    accuracy                           0.60        10
   macro avg       0.75      0.67      0.58        10
weighted avg       0.80      0.60      0.57        10

Confusion Matrix:
 [[4 0]
 [4 2]]
 Results saved to 'log_with_qml_predictions.csv'


**QNN**

In [36]:
import pennylane as qml
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

df = pd.read_csv("final_train.csv")
X = df.drop(columns=["level", "binary_attack"])
y = df["binary_attack"].map({'normal': 0, 'abnormal': 1})

categorical_cols = ["protocol_type", "service", "flag"]
numerical_cols = X.columns.difference(categorical_cols).tolist()

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

X_processed = preprocessor.fit_transform(X)
X_processed = X_processed.toarray() if hasattr(X_processed, "toarray") else X_processed

X_processed = X_processed[:10000]
y = y[:10000].values


pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_processed)

X_train_tensor = torch.tensor(X_pca, dtype=torch.float32)
y_train_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=2, shuffle=True)


n_qubits = 2
dev = qml.device("default.qubit", wires=n_qubits)


@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):

    qml.templates.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.templates.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))


class QNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.classical = nn.Linear(2, 2)
        self.q_params = nn.Parameter(0.01 * torch.randn((2, n_qubits)))

    def forward(self, x):
        x = self.classical(x)
        outputs = []
        for i in range(x.shape[0]):
            out = quantum_circuit(x[i], self.q_params)
            outputs.append(out)
        return torch.stack(outputs).unsqueeze(1)
model = QNN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.BCELoss()

epochs = 15
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()


        preds = torch.sigmoid(model(xb)).float()
        yb = yb.float()


        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss / len(train_loader):.4f}")


Epoch 1: Loss = 0.4239
Epoch 2: Loss = 0.4217
Epoch 3: Loss = 0.4218
Epoch 4: Loss = 0.4218
Epoch 5: Loss = 0.4217
Epoch 6: Loss = 0.4216
Epoch 7: Loss = 0.4217
Epoch 8: Loss = 0.4216
Epoch 9: Loss = 0.4214
Epoch 10: Loss = 0.4217
Epoch 11: Loss = 0.4219
Epoch 12: Loss = 0.4218
Epoch 13: Loss = 0.4216
Epoch 14: Loss = 0.4218
Epoch 15: Loss = 0.4214


In [ ]:
import joblib
joblib.dump(preprocessor, "qnn_preprocessor.pkl")
joblib.dump(pca, "qnn_pca.pkl")

print("Preprocessor and PCA for QNN saved.")


In [40]:
def save_model(model, path="qnn_model.pth"):
    torch.save(model.state_dict(), path)
    print(f"Model saved to {path}")

save_model(model, "qnn_model.pth")


def save_quantum_params(q_params, path="qnn_quantum_params.pth"):
    torch.save(q_params, path)
    print(f"Quantum parameters saved to {path}")

save_quantum_params(model.q_params, "qnn_quantum_params.pth")


Model saved to qnn_model.pth
Quantum parameters saved to qnn_quantum_params.pth


In [41]:
def load_quantum_params(path="qnn_quantum_params.pth"):
    return torch.load(path)

model = QNN()
model.load_state_dict(torch.load("qnn_model.pth"))
model.q_params = load_quantum_params("qnn_quantum_params.pth")
model.eval()


QNN(
  (classical): Linear(in_features=2, out_features=2, bias=True)
)

In [42]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pandas as pd

df_log = pd.read_csv("logfile12.csv")
df_original = pd.read_csv("final_test.csv").head(10)
y_true = df_original["binary_attack"].map({'normal': 0, 'abnormal': 1}).values

X_log = df_log.drop(columns=["level", "binary_attack"], errors="ignore")
X_log_processed = preprocessor.transform(X_log)
X_log_processed = X_log_processed.toarray() if hasattr(X_log_processed, "toarray") else X_log_processed
X_log_pca = pca.transform(X_log_processed)
X_log_tensor = torch.tensor(X_log_pca, dtype=torch.float32)

with torch.no_grad():
    outputs = torch.sigmoid(model(X_log_tensor)).squeeze()
    y_pred = (outputs >= 0.5).int().numpy()

accuracy = accuracy_score(y_true, y_pred)
print(f"QNN Model Accuracy: {accuracy:.4f}")
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=["normal", "abnormal"]))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

result_df = df_log.copy()
result_df["true_attack"] = ["abnormal" if y == 1 else "normal" for y in y_true]
result_df["predicted_attack"] = ["abnormal" if y == 1 else "normal" for y in y_pred]
result_df.to_csv("log_with_qnn_predictions.csv", index=False)
print(" Results saved to 'log_with_qnn_predictions.csv'")


QNN Model Accuracy: 0.6000

Classification Report:
               precision    recall  f1-score   support

      normal       0.50      1.00      0.67         4
    abnormal       1.00      0.33      0.50         6

    accuracy                           0.60        10
   macro avg       0.75      0.67      0.58        10
weighted avg       0.80      0.60      0.57        10

Confusion Matrix:
 [[4 0]
 [4 2]]
 Results saved to 'log_with_qnn_predictions.csv'
